# Поступашки: EDA и аудит качества данных (base.xlsx)

**Задача 1 хакатона.** Ниже — не набор графиков, а последовательный аудит:
по каждому пункту задания сначала фиксируется **проблема/вопрос**, затем
**находка** (что показали данные) и **решение** (как я договорился это
трактовать дальше по кейсу).

Пять пунктов аудита:
1. Что считать покупкой, заказом и уникальным покупателем.
2. Как обрабатывать одновременную покупку нескольких курсов и пакеты.
3. Повторные покупки, продуктовые сочетания, динамика продаж и выручки.
4. Временные закономерности, всплески, провалы, аномалии.
5. Какие бизнес-вопросы решаемы текущими данными, а какие — принципиально нет.

Файл: `base.xlsx` — 795 строк, поля: `student_id` (обезличенный id),
`amount` (сумма строки), `course` (курс), `ts` (время оплаты).

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
from scipy import stats

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

df = pd.read_excel('base.xlsx')
df.columns = ['student_id', 'amount', 'course', 'ts']
df['date'] = df['ts'].dt.date

print(df.shape)
print(df.dtypes)
df.head()

(795, 5)
student_id             int64
amount               float64
course                   str
ts            datetime64[us]
date                  object
dtype: object


   student_id  amount            course                  ts        date
0         437  8950.0            ML про 2026-08-04 09:08:31  2026-08-04
1         437  7890.0   Аналитика старт 2026-08-04 09:35:00  2026-08-04
2         443  8475.0  Линейная алгебра 2026-08-04 10:14:38  2026-08-04
3         443  8475.0        Мат анализ 2026-08-04 10:14:38  2026-08-04
4         430  8950.0            К ВУЗу 2026-08-04 13:38:06  2026-08-04

## 0. Технический аудит (предпосылка ко всему остальному)

Прежде чем говорить о бизнес-смысле, проверяем данные как таковые:
пропуски, полные дубли строк, диапазон дат, число уникальных студентов
и курсов.

In [2]:
print("Пропуски по колонкам:")
print(df.isna().sum())
print()
print("Полных дублей строк:", df.duplicated().sum())
print("Уникальных student_id:", df.student_id.nunique())
print("Уникальных курсов:", df.course.nunique())
print("Диапазон дат:", df.ts.min(), "—", df.ts.max())

Пропуски по колонкам:
student_id    0
amount        0
course        0
ts            0
date          0
dtype: int64

Полных дублей строк: 0
Уникальных student_id: 606
Уникальных курсов: 18
Диапазон дат: 2026-08-04 09:08:31 — 2026-09-10 06:09:12


In [3]:
df.course.value_counts()

course
Аналитика про          93
Аналитика старт        91
AI агенты              91
ML про                 90
Алгоритмы старт        88
ML старт               80
Backend про            55
Backend старт          39
Алгоритмы про          35
Алгоритмы              30
К ВУЗу                 26
Мат анализ             20
Линейная алгебра       19
Теория вероятностей    13
Data Science            9
АВ тестам               8
Data Engenering         4
Дискретка               4
Name: count, dtype: int64

## 1. Что считать покупкой, заказом и уникальным покупателем

**Проблема:** 795 строк файла — это не то же самое, что 795 "покупок".
Один платёж может состоять из нескольких строк (пакет курсов), а один
человек — совершить несколько разных платежей. Без явного определения
"заказа" все последующие метрики (выручка, AOV, конверсия) считаются
на глазок и разными людьми по-разному.

**Проверка:** группируем строки по `(student_id, ts)` — гипотеза в том,
что строки с одинаковым студентом и одинаковой секундой оплаты относятся
к одному платежу.

In [4]:
orders = df.groupby(['student_id', 'ts']).agg(
    amount=('amount', 'sum'),
    n_items=('course', 'count'),
    date=('date', 'first'),
).reset_index()

orders

     student_id                  ts  amount  n_items        date
0             1 2026-09-04 19:30:57  8950.0        1  2026-09-04
1             2 2026-09-05 16:22:48  8950.0        1  2026-09-05
2             3 2026-09-03 22:47:02  5975.0        1  2026-09-03
3             4 2026-08-31 15:42:34  8950.0        1  2026-08-31
4             5 2026-08-22 16:19:00  5750.0        1  2026-08-22
..          ...                 ...     ...      ...         ...
623         602 2026-09-05 17:42:39  8450.0        1  2026-09-05
624         603 2026-08-18 16:34:39  8745.0        1  2026-08-18
625         604 2026-09-06 19:17:57  8950.0        1  2026-09-06
626         605 2026-08-26 07:26:53  8950.0        1  2026-08-26
627         606 2026-08-10 18:36:00  6450.0        1  2026-08-10

[628 rows x 5 columns]

In [5]:
print("Строк в исходных данных:", len(df))
print("Заказов (уникальных student_id + ts):", len(orders))
print()
print("Распределение размера заказа (число курсов в одном платеже):")
print(orders.n_items.value_counts().sort_index())
print()
print("Заказов с >1 курсом (пакеты):", (orders.n_items > 1).sum())
print("Строк, входящих в пакеты:", df.groupby(['student_id', 'ts'])['course']
      .transform('size').gt(1).sum(), "из", len(df))

Строк в исходных данных: 795
Заказов (уникальных student_id + ts): 628

Распределение размера заказа (число курсов в одном платеже):
n_items
1    475
2    145
3      4
4      2
5      2
Name: count, dtype: int64

Заказов с >1 курсом (пакеты): 153
Строк, входящих в пакеты: 320 из 795


**Находка:** 795 строк → 628 заказов, 606 уникальных студентов. 153 из
628 заказов (24%) — пакеты из 2–5 курсов, это 320 строк (40% всех
строк).

**Решение:**
- **Покупка** = одна строка файла (один курс в составе платежа).
- **Заказ** = группа строк с одинаковыми `student_id` и `ts` (одна
  транзакция оплаты, независимо от числа курсов внутри).
- **Уникальный покупатель** = `student_id`.
- **Допущение**, которое я закладываю явно: секундной точности `ts`
  достаточно, чтобы не путать два разных заказа одного студента —
  при таком дневном объёме (максимум ~70 заказов/день на всю базу)
  коллизия по секундам у одного и того же студента маловероятна, но
  это предположение, а не доказанный факт, и его стоит проверить, если
  появится более точный лог.

## 2. Как обрабатывать одновременную покупку нескольких курсов и пакеты

**Проблема:** если считать `amount` ценой конкретного курса даже внутри
пакета, доход по каждому продукту будет посчитан неверно — нужно
понять, что вообще представляет собой сумма в пакетной строке.

**Проверка:** ищем дробные суммы (копейки) — если пакет продавался за
единую цену, а `amount` — это честная доля этой цены на курс, сумма по
строкам одного заказа должна сходиться в круглое число, а отдельные
строки — иметь копейки от деления.

In [6]:
frac_mask = (df['amount'] % 1 != 0)
print(f"Строк с дробной суммой (не целое число рублей): {frac_mask.sum()} из {len(df)}")
example = df[(df.student_id == 58)].sort_values('ts')
print()
print("Пример — студент 58, один платёж за 3 курса:")
print(example[['student_id', 'course', 'amount', 'ts']])
print()
print("Сумма по строкам:", example['amount'].sum(), "— целое число, хотя каждая строка дробная.")

Строк с дробной суммой (не целое число рублей): 21 из 795

Пример — студент 58, один платёж за 3 курса:
     student_id           course   amount                  ts
117          58    Backend старт  5463.33 2026-08-09 16:12:46
118          58         ML старт  5463.33 2026-08-09 16:12:46
119          58  Алгоритмы старт  5463.34 2026-08-09 16:12:46

Сумма по строкам: 16390.0 — целое число, хотя каждая строка дробная.


**Находка:** 5463.33 + 5463.33 + 5463.34 = 16390.00 — сумма заказа
поделена поровну на число курсов, остаток округления ушёл в последнюю
строку. Значит **amount пакетной строки — это доля суммы заказа, а
не цена конкретного курса** внутри пакета.

**Решение:**

Выручку и средний чек (AOV) считаем по заказу — то есть по сумме
всех строк одной покупки (student_id, ts), а не по каждой строке
отдельно. Иначе один платёж за несколько курсов посчитается как
несколько разных заказов, и средний чек занизится.

Сколько именно заработал каждый отдельный курс внутри пакета — не
считаем. В файле это просто "сумма заказа / число курсов", а не
настоящая цена курса, поэтому выдавать эту цифру за реальную выручку
курса нельзя — это будет придумано, а не измерено.

Для анализа "что покупают вместе" курсы внутри пакета всё равно
учитываем — сам факт совместной покупки реален, просто это не про
деньги по курсу, а про сочетание продуктов (см. п. 3).

## 3. Повторные покупки, продуктовые сочетания, динамика продаж и выручки

### 3.1 Повторные покупки

**Проблема:** нужно понять, насколько бизнес держится на повторных
клиентах — это влияет на то, стоит ли считать retention/LTV частью
будущей ROMI-модели.

In [7]:
per_student_orders = df.groupby('student_id')['ts'].nunique()
repeat_ids = per_student_orders[per_student_orders > 1].index

print("Распределение числа заказов на студента:")
print(per_student_orders.value_counts().sort_index())
print()
print(f"Повторных покупателей: {len(repeat_ids)} из {df.student_id.nunique()} "
      f"({len(repeat_ids) / df.student_id.nunique():.1%})")
rev_repeat = df[df.student_id.isin(repeat_ids)]['amount'].sum()
print(f"Их доля в общей сумме amount: {rev_repeat / df['amount'].sum():.1%}")

Распределение числа заказов на студента:
ts
1    586
2     19
4      1
Name: count, dtype: int64

Повторных покупателей: 20 из 606 (3.3%)
Их доля в общей сумме amount: 6.3%


**Находка:** 20 из 606 студентов (3.3%) сделали больше одного заказа,
на них приходится 6.3% суммарного `amount`.

**Решение:** повторная покупка — редкое исключение. На нынешнем объёме
данных **не закладываю retention/LTV как основу ROMI-модели** — с 20
наблюдениями это статистически ненадёжно.

### 3.2 Продуктовые сочетания

**Проблема:** нужно понять, случайны ли пакеты или это осознанная
продуктовая логика (апселл/кросс-селл) — это влияет на то, как в
будущем считать ROMI кампании, которая рекламирует один курс, а
покупают два.

In [8]:
combo_counter = Counter()
for (sid, ts), g in df.groupby(['student_id', 'ts']):
    if len(g) > 1:
        for pair in combinations(sorted(g.course.tolist()), 2):
            combo_counter[pair] += 1

for pair, cnt in combo_counter.most_common(10):
    print(cnt, pair)

24 ('ML старт', 'Алгоритмы старт')
23 ('Аналитика про', 'Аналитика старт')
18 ('AI агенты', 'ML про')
13 ('ML про', 'ML старт')
10 ('Алгоритмы старт', 'Аналитика старт')
10 ('Алгоритмы про', 'Алгоритмы старт')
9 ('Backend старт', 'Алгоритмы старт')
6 ('AI агенты', 'Backend про')
5 ('Линейная алгебра', 'Мат анализ')
5 ('Backend про', 'Backend старт')


**Находка:** топ-связки — не случайны: "старт + про" одного трека
(апселл: Аналитика, Backend, Алгоритмы) и смежные треки (ML + Алгоритмы,
AI агенты + ML).

**Решение:** пакеты — это продуктовая механика (апселл/кросс-селл), а
не шум. При построении attribution/ROMI (Задачи 5–8) заказ нужно
считать по его полному составу курсов, а не только по "первому"
курсу: выручка одного рекламного касания часто равна не одному курсу,
а сумме пакета.

### 3.3 Динамика продаж и выручки

In [9]:
daily = orders.groupby('date').agg(
    orders=('student_id', 'count'),
    revenue=('amount', 'sum'),
    items=('n_items', 'sum'),
).reset_index()
daily['dow'] = pd.to_datetime(daily['date']).dt.day_name()
daily

          date  orders    revenue  items        dow
0   2026-08-04       7   84130.00      9    Tuesday
1   2026-08-05       6   67110.00      8  Wednesday
2   2026-08-06       7   62445.00      8   Thursday
3   2026-08-07       5   54995.00      6     Friday
4   2026-08-08      35  275015.00     49   Saturday
5   2026-08-09      68  524091.67     92     Sunday
6   2026-08-10      29  205590.00     36     Monday
7   2026-08-11      12   96330.00     17    Tuesday
8   2026-08-12      14  114930.00     17  Wednesday
9   2026-08-13       9   72810.00      9   Thursday
10  2026-08-14       9   76690.00     10     Friday
11  2026-08-15       5   48345.00      7   Saturday
12  2026-08-16       5   43085.00      6     Sunday
13  2026-08-17       3   33185.00      4     Monday
14  2026-08-18       6   61110.00      7    Tuesday
15  2026-08-19       5   56380.00      7  Wednesday
16  2026-08-20       6   52585.00      6   Thursday
17  2026-08-21       3   28145.00      3     Friday
18  2026-08-

In [10]:
order_totals = orders['amount']
print("Общая выручка (по заказам, без задвоения пакетов):",
      round(order_totals.sum(), 2))
print("Средний чек (AOV):", round(order_totals.mean(), 1))
print("Медианный чек:", order_totals.median())

Общая выручка (по заказам, без задвоения пакетов): 5904671.67
Средний чек (AOV): 9402.3
Медианный чек: 8950.0


**Находка:** выручка растёт неравномерно, с выраженными недельными
волнами (детали и статистическая проверка — в п. 4). AOV ≈ 9 402 ₽,
медиана 8 950 ₽ — совпадает с "базовой" ценой одного курса, то есть
типичный заказ — это один курс по полной цене, а пакеты и скидки
двигают среднее, но не медиану.

**Решение:** как целевую метрику продаж для дальнейших задач (прогноз,
ROMI) беру **выручку по заказам**, а не количество заказов как
таковых, иначе рост числа пакетов будет выглядеть как рост продаж без
реального роста выручки.

## 4. Временные закономерности, всплески, провалы, аномалии

### 4.1 Всплески по дням

In [11]:
daily

          date  orders    revenue  items        dow
0   2026-08-04       7   84130.00      9    Tuesday
1   2026-08-05       6   67110.00      8  Wednesday
2   2026-08-06       7   62445.00      8   Thursday
3   2026-08-07       5   54995.00      6     Friday
4   2026-08-08      35  275015.00     49   Saturday
5   2026-08-09      68  524091.67     92     Sunday
6   2026-08-10      29  205590.00     36     Monday
7   2026-08-11      12   96330.00     17    Tuesday
8   2026-08-12      14  114930.00     17  Wednesday
9   2026-08-13       9   72810.00      9   Thursday
10  2026-08-14       9   76690.00     10     Friday
11  2026-08-15       5   48345.00      7   Saturday
12  2026-08-16       5   43085.00      6     Sunday
13  2026-08-17       3   33185.00      4     Monday
14  2026-08-18       6   61110.00      7    Tuesday
15  2026-08-19       5   56380.00      7  Wednesday
16  2026-08-20       6   52585.00      6   Thursday
17  2026-08-21       3   28145.00      3     Friday
18  2026-08-

Видно три чётких всплеска, каждый выпадает на выходные:

- 08–10 авг: 35 → 68 → 29 заказов (норма буднего дня — 5–15)
- 22–24 авг: 26 → 39 → 19 заказов
- 04–06 сен: 26 → 49 → 45 заказов

Интервал между пиками — около двух недель. Провалов (аномальных
падений среди полных дней) не обнаружено — все 38 календарных дней
периода присутствуют в данных без пропусков. Последний день
(10 сентября) не провал, а технический обрыв — данные заканчиваются в
06:09, то есть это неполные сутки, и его нужно исключать из сравнений
динамики, а не трактовать как падение спроса.

Проверяем эффект выходных формально (t-test Уэлча, т.к. дисперсии в
группах разные).

**H0: разницы между количеством заказов в выходные и в будни нет**

**H1: разница есть**

In [12]:
daily_full = daily[daily['date'] != daily['date'].max()].copy()
daily_full['is_weekend'] = pd.to_datetime(daily_full['date']).dt.dayofweek >= 5

weekend = daily_full.loc[daily_full.is_weekend, 'orders']
weekday = daily_full.loc[~daily_full.is_weekend, 'orders']

t_stat, p_val = stats.ttest_ind(weekend, weekday, equal_var=False)
df_welch = (weekend.var() / len(weekend) + weekday.var() / len(weekday)) ** 2 / (
    (weekend.var() / len(weekend)) ** 2 / (len(weekend) - 1)
    + (weekday.var() / len(weekday)) ** 2 / (len(weekday) - 1)
)
t_crit = stats.t.ppf(0.975, df_welch)

print(f"Выходные: n={len(weekend)}, mean={weekend.mean():.1f}, sd={weekend.std():.1f}")
print(f"Будни:    n={len(weekday)}, mean={weekday.mean():.1f}, sd={weekday.std():.1f}")
print(f"Welch t-test: t={t_stat:.2f}, df={df_welch:.1f}, "
      f"t_crit(α=0.05, двусторонний)={t_crit:.2f}, p={p_val:.5f}")

Выходные: n=10, mean=30.1, sd=20.9
Будни:    n=27, mean=12.1, sd=7.3
Welch t-test: t=2.67, df=9.8, t_crit(α=0.05, двусторонний)=2.23, p=0.02381


|t| = 2.67 > t_crit = 2.23 → различие значимо на уровне 5% (p ≈ 0.024).

"Выходные/будни" и "двухнедельный цикл" - рабочая гипотеза источника
всплесков (лончи или промо-рассылки в канале, таргетированные на
выходные), а не факт, потому что прямого журнала кампаний в этих
данных нет (проверить её — задача восстановления маркетинговой
истории, Задача 2).

### 4.2 Аномалии в цене

In [13]:
print("Общая статистика по amount:")
print(df['amount'].describe())
print()
print("15 самых низких сумм:")
print(df.nsmallest(15, 'amount')[['student_id', 'amount', 'course', 'ts']])
print()
print("10 самых высоких сумм:")
print(df.nlargest(10, 'amount')[['student_id', 'amount', 'course', 'ts']])

Общая статистика по amount:
count      795.000000
mean      7427.259962
std       1827.497664
min        500.000000
25%       6490.000000
50%       7475.000000
75%       8950.000000
max      19350.000000
Name: amount, dtype: float64

15 самых низких сумм:
     student_id  amount               course                  ts
23          477   500.0  Теория вероятностей 2026-08-06 14:52:49
134         462   500.0      Алгоритмы старт 2026-08-09 16:45:22
486         437  1000.0            AI агенты 2026-08-30 16:16:42
714         366  2237.5            AI агенты 2026-09-06 15:18:37
715         366  2237.5               ML про 2026-09-06 15:18:37
716         366  2237.5             ML старт 2026-09-06 15:18:37
717         366  2237.5      Алгоритмы старт 2026-09-06 15:18:37
719          55  2237.5            AI агенты 2026-09-06 15:35:13
720          23  2237.5             ML старт 2026-09-06 15:39:44
721          51  2237.5               ML про 2026-09-06 15:46:24
724         180  2237.5      

**Находка / решение по каждой аномалии** (фиксирую как открытые вопросы
к бизнесу, а не тихо чищу или додумываю):

- **Кластер 2237.5 ₽** — 6 позиций (ML старт, ML про, AI агенты,
  Алгоритмы старт), все с 6 сентября 15:18 до 16:23. Один и тот же
  нестандартный ценник на разные продукты в узком часовом окне —
  разовая флеш-акция/промокод. Это пример того, как по всплеску цены
  можно частично восстановить маркетинговую активность даже без
  прямого журнала рекламы (полезно для Задачи 2).
- **500 ₽** (2 платежа) — на порядок ниже минимальной цены курса
  (~2200 ₽ на всё остальное). Тест/сотрудник/промокод/ошибка — не
  определить по имеющимся данным.
- **Верхние выбросы**: 19 350 ₽ (Алгоритмы, один курс), 18 900 ₽ × 2
  (Теория вероятностей), 15 695 ₽ (Data Science) — не совпадают ни с
  одной известной комбинацией пакета.
- Цена одного и того же курса в принципе очень нестабильна (пример:
  «Алгоритмы» — от 4225 ₽ до 19 350 ₽). Решение: не считать "цену
  курса" константой ни в одной последующей модели — только фактическую
  сумму заказа.

## 5. Какие бизнес-вопросы решаемы текущими данными, а какие — нет

**Решаемо этими данными:**
- объём и динамика продаж/выручки по дням, включая статистически
  подтверждённые всплески;
- состав пакетов и апселл/кросс-селл связки между курсами;
- доля и вклад повторных покупателей;
- средний чек и его разброс;
- обнаружение подозрительных ценовых кластеров как косвенных следов
  промо-активности (без подтверждения дат из внешнего источника).

**Принципиально нерешаемо этими данными (нужны Задачи 2–5):**
- какая покупка от какого рекламного канала/размещения/креатива
  пришла — в `base.xlsx` нет ключа к источнику трафика вообще;
- ROMI и любая атрибуция — физически не на чем считать без данных о
  рекламных касаниях, стоимости размещений и хотя бы приблизительного
  tracking-ключа, связывающего рекламу с `student_id`;
- инкрементальность (что реклама добавила сверх органики) — без
  контрольной группы или эксперимента attribution ≠ incrementality
  (Задача 7), а способа его провести на исторических данных нет.

---

## 6. Задача 2 — сопоставление с маркетинговой историей

Ниже — не новый источник данных, а проверка гипотезы из п.4.1
("возможно, лончи/промо по выходным") на реальных постах, которые
удалось восстановить вручную из публичного Telegram (`t.me/s/postypashki_old`)
и TGStat (`marketing_posts.csv`, лежит рядом с этим ноутбуком).

**Важная оговорка про метод**, чтобы не выдать желаемое за
измеренное: список постов в этом файле — не полная лента канала
день за днём, а **целенаправленно найденные** посты вокруг уже
известных из п.4.1 всплесков продаж. Поэтому сравнение
"есть пост / нет поста" ниже — это не независимая проверка гипотезы
"посты вызывают продажи", а **количественное подтверждение размера**
уже найденной связи. Причинность отсюда не следует — только то, что
величина связи не в пределах шума.

### 6.1 Загрузка кампаний

In [14]:
posts = pd.read_csv('marketing_posts.csv')
posts['date'] = pd.to_datetime(posts['date']).dt.date
posts_campaign = posts[posts['campaign_id'] != 'none'].copy()

print(posts.shape, "постов всего,", len(posts_campaign), "относятся к кампаниям")
posts_campaign[['campaign_id', 'date', 'type', 'views', 'clicks_fwd']]

(17, 14) постов всего, 15 относятся к кампаниям


           campaign_id        date              type    views  clicks_fwd
0           start_sale  2026-08-08       sale_launch  26900.0        98.0
3           start_sale  2026-08-10     sale_extended  16400.0         8.0
4           pro_launch  2026-08-22            launch  35400.0       152.0
5       tbank_deadline  2026-08-31   content_trigger  20500.0       316.0
6       tbank_deadline  2026-09-01   engagement_hook  17200.0       364.0
7       tbank_deadline  2026-09-02   content_trigger  16900.0       272.0
8       tbank_deadline  2026-09-03           urgency  14900.0       298.0
9       tbank_deadline  2026-09-05           unknown  15300.0        50.0
10      tbank_deadline  2026-09-06           urgency  12500.0       246.0
11  partner_crosspromo  2026-09-06       lead_magnet  14400.0        94.0
12      tbank_deadline  2026-09-06           content  11800.0       239.0
13      tbank_deadline  2026-09-06      sale_content  10900.0       442.0
14      tbank_deadline  2026-09-06    

### 6.2 Выручка в дни с кампанией vs без неё

In [15]:
campaign_days = set(posts_campaign['date'])
daily_full['has_campaign_post'] = daily_full['date'].isin(campaign_days)

with_post = daily_full.loc[daily_full.has_campaign_post, 'revenue']
without_post = daily_full.loc[~daily_full.has_campaign_post, 'revenue']

t_stat2, p_val2 = stats.ttest_ind(with_post, without_post, equal_var=False)
df_welch2 = (with_post.var() / len(with_post) + without_post.var() / len(without_post)) ** 2 / (
    (with_post.var() / len(with_post)) ** 2 / (len(with_post) - 1)
    + (without_post.var() / len(without_post)) ** 2 / (len(without_post) - 1)
)
t_crit2 = stats.t.ppf(0.975, df_welch2)

print(f"Дни с постом кампании:  n={len(with_post)}, "
      f"mean revenue={with_post.mean():.0f} ₽, sd={with_post.std():.0f}")
print(f"Дни без поста:          n={len(without_post)}, "
      f"mean revenue={without_post.mean():.0f} ₽, sd={without_post.std():.0f}")
print(f"Welch t-test: t={t_stat2:.2f}, df={df_welch2:.1f}, "
      f"t_crit(α=0.05, двусторонний)={t_crit2:.2f}, p={p_val2:.5f}")

Дни с постом кампании:  n=10, mean revenue=261664 ₽, sd=101047
Дни без поста:          n=27, mean revenue=121411 ₽, sd=114075
Welch t-test: t=3.62, df=18.1, t_crit(α=0.05, двусторонний)=2.10, p=0.00195


**Находка:** средняя дневная выручка в дни с найденным постом кампании
выше, чем в остальные дни, и разница статистически значима
(|t| > t_crit). Это ожидаемо — именно вокруг этих дат мы и искали
посты, — но теперь у нас есть число, а не "на глаз похоже".

**Решение:** использовать этот результат как **подтверждение**
гипотезы из п.4.1, а не как её независимое доказательство. В
презентации формулировать аккуратно: "выручка в дни известных кампаний
статистически выше базовой" — не "реклама вызывает продажи".

### 6.3 Охват кампании vs прирост выручки в её окне

**Проблема:** охват (просмотры/переходы) и выручка измеряются в разных
единицах и по разным датам — просмотры у поста, выручка у заказа.
Нужен способ их сопоставить, не выдумывая atrribution на уровне
пользователя (мы уже установили в предыдущем разделе, что это
невозможно на этих данных).

**Решение:** сравнивать не пользователей, а **окна**: для каждой из 3
известных кампаний берём (а) суммарный охват её постов (views + клики)
и (б) фактическую выручку в дни всплеска из п.4.1 минус базовая
(медианная) выручка обычного дня — это и есть грубая оценка "прироста"
за счёт кампании, при условии что в этом окне не было других
конкурирующих объяснений.

In [16]:
baseline_days = daily_full.loc[~daily_full.has_campaign_post, 'revenue']
baseline_daily_revenue = baseline_days.median()

spike_windows = {
    'start_sale': (pd.to_datetime('2026-08-08').date(), pd.to_datetime('2026-08-10').date()),
    'pro_launch': (pd.to_datetime('2026-08-22').date(), pd.to_datetime('2026-08-24').date()),
    'tbank_deadline': (pd.to_datetime('2026-09-04').date(), pd.to_datetime('2026-09-06').date()),
}

rows = []
for camp, (start, end) in spike_windows.items():
    window_revenue = daily_full.loc[
        (daily_full['date'] >= start) & (daily_full['date'] <= end), 'revenue'
    ].sum()
    n_days = (end - start).days + 1
    uplift = window_revenue - baseline_daily_revenue * n_days

    # для tbank_deadline считаем охват вместе с partner_crosspromo —
    # это один и тот же информационный повод (пост 6 сен), а не
    # отдельная кампания
    camp_ids = [camp] if camp != 'tbank_deadline' else ['tbank_deadline', 'partner_crosspromo']
    reach = posts_campaign.loc[
        posts_campaign['campaign_id'].isin(camp_ids), ['views', 'clicks_fwd']
    ].sum().sum()

    rows.append({
        'campaign': camp,
        'window': f"{start}..{end}",
        'reach_views_clicks': int(reach),
        'window_revenue': round(window_revenue),
        'baseline_revenue_for_window': round(baseline_daily_revenue * n_days),
        'uplift_revenue': round(uplift),
        'uplift_per_1000_reach': round(uplift / (reach / 1000), 1) if reach else None,
    })

campaign_summary = pd.DataFrame(rows)
campaign_summary

         campaign                  window  reach_views_clicks  window_revenue  baseline_revenue_for_window  uplift_revenue  uplift_per_1000_reach
0      start_sale  2026-08-08..2026-08-10               43406         1004697                       230070          774627                17846.1
1      pro_launch  2026-08-22..2026-08-24               35552          829718                       230070          599648                16866.8
2  tbank_deadline  2026-09-04..2026-09-06              166471         1150344                       230070          920274                 5528.1

**Находка:** во всех трёх окнах фактическая выручка заметно выше
базовой (положительный `uplift_revenue`), а `uplift_per_1000_reach`
даёт сопоставимую по кампаниям "эффективность охвата" — сколько рублей
прироста приходится на 1000 просмотров/переходов. Это позволяет
сравнивать кампании между собой даже без единого рубля данных о
стоимости размещения.

**Решение и явная граница метода:** `uplift_per_1000_reach` — это
**не ROMI**. Явно проговариваем разницу дальше.

### 6.4 Как считать ROMI — что можно сейчас и чего не хватает

**Целевая формула (задана в кейсе):**

```
ROMI = (Attributed Revenue − Marketing Cost) / Marketing Cost × 100%
```

**Проблема:** у нас нет ни одного из двух слагаемых в чистом виде.
- `Attributed Revenue` требует attribution на уровне пользователя
  (клик/касание → лид → оплата), а мы установили в разделе про
  атрибуцию (после Задачи 2), что связать `student_id` с конкретным
  постом/каналом невозможно — нет tracking-ключа и нет права сшивать
  анонимный id с реальным Telegram-аккаунтом.
- `Marketing Cost` отсутствует полностью: все найденные кампании — это
  посты в собственных каналах (не платное размещение), у них нет
  медиа-бюджета, а издержки на производство контента исторически не
  логировались. Найти реальное **платное** внешнее размещение с
  известной ценой за период кейса не удалось (см. раздел про сторонние
  каналы) — соответственно, для 100% найденных кампаний знаменатель
  формулы буквально неоткуда взять, а не "посчитан неточно".

**Решение:**
1. Сейчас вместо ROMI считаем прокси — `uplift_per_1000_reach` из 6.3.
   Это направленный (не денежный) показатель эффективности, годится
   для сравнения кампаний между собой ("охват вокруг ПРО-лонча
   конвертировался в выручку лучше/хуже, чем охват вокруг СТАРТа"), но
   **не годится** как ответ на "куда направить следующий бюджет в
   рублях" — это и есть главный бизнес-вопрос кейса (слайд 8), и
   честный ответ на него сегодня: "нельзя посчитать, вот почему".
2. Чтобы формула заработала в будущем (зона Задачи 5), в модель данных
   должны попасть **до** запуска кампании, а не восстанавливаться
   постфактум:
   - `cost` и `publication_time` на уровне каждого `placement` — в том
     числе для постов в собственных каналах (учётная, а не рыночная
     стоимость: например, часы автора/дизайнера);
   - уникальный `campaign_id`/`placement_id`/`creative_id` в каждой
     ссылке, которую пользователь видит в посте (это уже прямо
     сформулировано в Задаче 5 кейса);
   - join-ключ от клика по этой ссылке до `payment`, то есть решённый
     user stitching (Задача 4) — без него `Attributed Revenue`
     физически не из чего собрать.

Итог для передачи в Задачи 3–5: ROMI по историческим данным — это
**нерешаемая задача**, а не "мы не успели посчитать". Всё, что можно
сделать без выдумывания — это `uplift_per_1000_reach`, использовать
дальше только его, и не подставлять эту цифру туда, где кейс просит
именно ROMI.

---

## 7. MVP: рабочий прототип attribution + ROMI (Задачи 6, 7, 8)

Раздел 6 показал, что ROMI **на реальных цифрах прошлого** посчитать
нельзя — ни числителя (Attributed/Incremental Revenue на уровне
пользователя), ни знаменателя (Marketing Cost) взять неоткуда. Но
кейс (слайд 20, "MVP обязателен") просит не готовую цифру, а
**рабочую систему**: код, который решает реальный кусок цепочки
измерения и явно помечает, где данные настоящие, а где — синтетика
для демонстрации логики. Это разные требования, и ниже — именно
второе.

**Что в этом разделе реально, а что синтетика:**
- `uplift_revenue` (числитель ROMI_inc) — **реальные данные**, взято
  из раздела 6.3 без изменений.
- Touch-уровневые данные (кто, по какой ссылке, когда кликнул) —
  **синтетика**: таких данных исторически нет вообще, они нужны только
  чтобы показать, как работает сама attribution-модель.
- Marketing Cost для ROMI_attr — гибрид: число постов кампании
  реальное (`marketing_posts.csv`), а часы на один пост и внутренняя
  ставка автора — SYNTHETIC-допущение (это никогда не логировалось).
  Для ROMI_inc (реальный `uplift_revenue`) cost вообще не передаём —
  функция честно возвращает `None`, а не 0 и не выдумку.

### 7.1 Attribution model (Задача 6)

**Решение по модели:** беру **last-touch в окне 7 дней**. Обоснование:
у нас нет ни одного реального примера мультитач-пути (одновременных
разных касаний одного человека), поэтому любая модель сложнее
last-touch (linear, time-decay, position-based) распределяла бы
выручку по весам, которые мы бы просто придумали — а слайд 15 прямо
предупреждает: сложная модель не значит лучшая. Окно 7 дней — по
наблюдаемому в разделе 6 лагу "пост кампании → рост выручки" (1-3 дня
до пика, беру с запасом).

In [17]:
def attribution_model(touches, purchase_ts, purchase_revenue, model='last_touch', window_days=7):
    """
    touches: список (channel, touch_ts) для одного покупателя.
    Возвращает {channel: attributed_revenue} по выбранной модели,
    учитывая только касания в пределах window_days до покупки.
    """
    window_start = purchase_ts - pd.Timedelta(days=window_days)
    eligible = [(ch, ts) for ch, ts in touches if window_start <= ts <= purchase_ts]
    if not eligible:
        return {'organic_or_out_of_window': purchase_revenue}

    eligible.sort(key=lambda x: x[1])
    result = {}
    if model == 'last_touch':
        result[eligible[-1][0]] = purchase_revenue
    elif model == 'first_touch':
        result[eligible[0][0]] = purchase_revenue
    elif model == 'linear':
        share = purchase_revenue / len(eligible)
        for ch, _ in eligible:
            result[ch] = result.get(ch, 0) + share
    else:
        raise ValueError(f"неизвестная модель: {model}")
    return result

### 7.2 Синтетический пример на нескольких моделях

Ниже — придуманные (SYNTHETIC) покупатели и их касания, только чтобы
показать: выбор модели реально меняет, какому каналу "достаётся"
выручка. Каналы взяты из реальных `campaign_id` раздела 6, суммы и
пути — синтетика.

In [18]:
np.random.seed(42)  # SYNTHETIC ниже — фиксирую сид для воспроизводимости примера

synthetic_customers = [
    {  # мультитач: увидел лонч ПРО, потом дедлайн Т-Банка, купил
        'revenue': 14795,
        'purchase_ts': pd.Timestamp('2026-09-06 20:00'),
        'touches': [('pro_launch', pd.Timestamp('2026-08-22 13:19')),
                    ('tbank_deadline', pd.Timestamp('2026-09-03 19:06'))],
    },
    {  # одно касание — старт-распродажа
        'revenue': 6490,
        'purchase_ts': pd.Timestamp('2026-08-09 10:00'),
        'touches': [('start_sale', pd.Timestamp('2026-08-08 12:13'))],
    },
    {  # три касания подряд перед покупкой в дедлайн
        'revenue': 8950,
        'purchase_ts': pd.Timestamp('2026-09-06 21:00'),
        'touches': [('tbank_deadline', pd.Timestamp('2026-09-01 20:40')),
                    ('tbank_deadline', pd.Timestamp('2026-09-03 19:06')),
                    ('partner_crosspromo', pd.Timestamp('2026-09-06 15:06'))],
    },
    {  # покупка вне 7-дневного окна любого касания -> органика
        'revenue': 9990,
        'purchase_ts': pd.Timestamp('2026-08-30 12:00'),
        'touches': [('pro_launch', pd.Timestamp('2026-08-22 13:19'))],
    },
]

comparison = {}
for model in ['first_touch', 'last_touch', 'linear']:
    totals = Counter()
    for cust in synthetic_customers:
        attributed = attribution_model(
            cust['touches'], cust['purchase_ts'], cust['revenue'], model=model
        )
        for ch, rev in attributed.items():
            totals[ch] += rev
    comparison[model] = dict(totals)

pd.DataFrame(comparison).fillna(0)

                          first_touch  last_touch        linear
tbank_deadline                23745.0       14795  20761.666667
start_sale                     6490.0        6490   6490.000000
organic_or_out_of_window       9990.0        9990   9990.000000
partner_crosspromo                0.0        8950   2983.333333

**Находка (на синтетике):** first-touch отдаёт всю выручку 3-го
покупателя каналу `tbank_deadline` (первое касание), last-touch —
каналу `partner_crosspromo` (последнее касание перед покупкой), linear
делит между обоими. Разница по каналу может быть в разы — это
буквально то, о чём предупреждает слайд 15 ("сравните подходы").

**Решение:** в проде фиксируем last-touch/7 дней как стартовую модель
(см. 7.1), но код держим модель-агностичным (`model=...` параметр) —
чтобы поменять на linear/position-based одной строкой, когда появятся
реальные touch-данные.

### 7.3 ROMI_attr и ROMI_inc — рабочие функции

**Метод расчёта cost (для ROMI_attr ниже):** размещения — это посты в
собственных каналах, а не платная реклама, поэтому cost — это
альтернативные издержки на создание поста (см. п. 6.4, вариант 2):
`cost = число_постов × часы_на_пост × ставка_часа`. Число постов
кампании берём **реальное** (`marketing_posts.csv`), а часы на один
пост и внутреннюю ставку автора — как **SYNTHETIC**-допущение, потому
что это никогда не логировалось.

In [19]:
def compute_romi(attributed_or_incremental_revenue, marketing_cost):
    """
    ROMI = (Revenue - Cost) / Cost.
    Если cost неизвестен — возвращаем None, а не 0 и не выдумку.
    """
    if marketing_cost is None or marketing_cost == 0:
        return None
    return (attributed_or_incremental_revenue - marketing_cost) / marketing_cost

# ROMI_attr — на синтетических touch-данных 7.2. Cost = n_posts (РЕАЛЬНЫЕ,
# из marketing_posts.csv) × часы_на_пост × ставка_часа (SYNTHETIC-допущения)
n_posts_by_channel = posts_campaign['campaign_id'].value_counts().to_dict()

HOURS_PER_POST = 1.5  # SYNTHETIC: часы на текст + картинку + согласование
HOURLY_RATE = 800      # SYNTHETIC: внутренняя ставка автора/smm, ₽/час

synthetic_cost_by_channel = {
    ch: round(n * HOURS_PER_POST * HOURLY_RATE)
    for ch, n in n_posts_by_channel.items()
}
print("Cost по каналам (n_posts — реальные, часы и ставка — SYNTHETIC):")
print(synthetic_cost_by_channel)

romi_attr_example = {
    ch: compute_romi(rev, synthetic_cost_by_channel.get(ch))
    for ch, rev in comparison['last_touch'].items() if ch != 'organic_or_out_of_window'
}
print()
print("ROMI_attr (revenue — на СИНТЕТИЧЕСКИХ touch-данных 7.2, cost — по методу "
      "'часы×ставка' выше, только для проверки логики):")
print(romi_attr_example)

# ROMI_inc, вариант A — на РЕАЛЬНОМ uplift_revenue из 6.3, cost неизвестен:
# функция должна честно отказаться считать, а не подставить 0.
print()
print("ROMI_inc, если cost НЕ известен (revenue — РЕАЛЬНЫЕ данные из 6.3):")
for _, row in campaign_summary.iterrows():
    romi_inc = compute_romi(row['uplift_revenue'], marketing_cost=None)
    print(f"  {row['campaign']}: incremental_revenue={row['uplift_revenue']} ₽, "
          f"cost=неизвестен -> ROMI_inc={romi_inc}")

# ROMI_inc, вариант B — та же РЕАЛЬНАЯ revenue, но cost оценён тем же
# гибридным методом, что и для ROMI_attr выше (n_posts РЕАЛЬНЫЕ,
# часы/ставка SYNTHETIC-допущение). Это не измеренный ROMI, а оценка
# при явно названном допущении — но именно она пригодна для сравнения
# кампаний между собой при распределении будущего бюджета.
print()
print("ROMI_inc, если cost ОЦЕНЁН по методу 'часы×ставка' (revenue реальная, cost synthetic):")
for _, row in campaign_summary.iterrows():
    camp_ids = [row['campaign']] if row['campaign'] != 'tbank_deadline' \
        else ['tbank_deadline', 'partner_crosspromo']
    cost_estimate = sum(synthetic_cost_by_channel.get(c, 0) for c in camp_ids)
    romi_inc_est = compute_romi(row['uplift_revenue'], cost_estimate)
    print(f"  {row['campaign']}: incremental_revenue={row['uplift_revenue']} ₽ (реальные), "
          f"cost≈{cost_estimate} ₽ (synthetic) -> ROMI_inc≈{romi_inc_est:.2f}")

Cost по каналам (n_posts — реальные, часы и ставка — SYNTHETIC):
{'tbank_deadline': 12000, 'start_sale': 2400, 'pro_launch': 1200, 'partner_crosspromo': 1200, 'free_week': 1200}

ROMI_attr (revenue — на СИНТЕТИЧЕСКИХ touch-данных 7.2, cost — по методу 'часы×ставка' выше, только для проверки логики):
{'tbank_deadline': 0.23291666666666666, 'start_sale': 1.7041666666666666, 'partner_crosspromo': 6.458333333333333}

ROMI_inc, если cost НЕ известен (revenue — РЕАЛЬНЫЕ данные из 6.3):
  start_sale: incremental_revenue=774627 ₽, cost=неизвестен -> ROMI_inc=None
  pro_launch: incremental_revenue=599648 ₽, cost=неизвестен -> ROMI_inc=None
  tbank_deadline: incremental_revenue=920274 ₽, cost=неизвестен -> ROMI_inc=None

ROMI_inc, если cost ОЦЕНЁН по методу 'часы×ставка' (revenue реальная, cost synthetic):
  start_sale: incremental_revenue=774627 ₽ (реальные), cost≈2400 ₽ (synthetic) -> ROMI_inc≈321.76
  pro_launch: incremental_revenue=599648 ₽ (реальные), cost≈1200 ₽ (synthetic) -> ROMI_inc≈498

**Решение:** функция `compute_romi` показывает оба варианта на одной и
той же реальной `campaign_summary.uplift_revenue`:
- **cost не передан** → честно `None`, а не 0 (0 дал бы
  бесконечный/бессмысленный ROMI) и не выдуманное число;
- **cost передан как явно названное допущение** (гибридный метод
  "часы × ставка" из начала 7.3, где число постов — реальное) →
  функция без изменений в коде считает оценочный `ROMI_inc`, который
  уже можно использовать, чтобы сравнить кампании между собой при
  распределении 300 000 ₽ (главный бизнес-вопрос кейса) — не как
  единственно верную цифру, а как систему с явно проговоренными
  допущениями.

Как только бизнес начнёт реально логировать `cost` по Задаче 5 — та же
функция посчитает точный `ROMI_inc` без единой правки в коде.

### 7.4 Incrementality (Задача 7) — что уже есть и что предложить дальше

**Attribution ≠ Incrementality** (слайд 16): attribution в 7.1-7.3
отвечает "кому мы приписали продажу", а не "случилась ли она вообще
благодаря рекламе". Наш `uplift_revenue` из 6.3 — это простейший
причинный дизайн из допустимого на слайде 16 списка: **interrupted
time series** (сравнение факта с базовой линией "обычного дня"). Он
реальный и уже посчитан — специально ставить дополнительный A/B-тест
на данные прошлого не нужно и невозможно: прошлое нельзя
рандомизировать задним числом.

**Что предложить на будущее** — и здесь как раз пригождается находка
про 5+ собственных каналов Поступашек (раздел про сеть каналов):
раз весь трафик — свой, можно поставить **randomized holdout** почти
бесплатно, без внешнего вендора:
- на следующий лонч случайно не показывать акцию ~15-20% аудитории
  одного из каналов (или использовать один канал сети как контроль, а
  структурно похожий — как treatment);
- сравнить конверсию/выручку в holdout-группе с остальными Welch
  t-test-ом — тем же методом, что уже применялся в разделе 4.1;
- это даст настоящий ROMI_inc с числителем, посчитанным не по
  interrupted time series (слабый дизайн), а по рандомизированному
  эксперименту (сильный дизайн) — то, что слайд 16 называет лучшим
  вариантом из списка.